<a href="https://colab.research.google.com/github/BryanHinostroza/lab04-bh/blob/develop/Laboratorio_4_1_MD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#LABORATORIO 4: INTEGRACIÓN Y LIMPIEZA. TABLA DE VARIABLES Y CREACIÓN DE NUEVAS VARIABLES. TRANSFORMACIÓN DE DATOS

## INTEGRANTES:
### - Gutierrez Garcia, Angela Belen
### - Hinostroza Martinez, Bryan Jesus
### - Huaman Quispe, Abraham Sebastian Jonathan

## A.- Por medio de la librería ‘Pandas’, lea la base de datos, asigne nombres para las columnas, realice una imputación para la variable ‘Bare Nuclei’ por medio de la mediana y cambie las categorías para la variable ‘Class’ de 2 o 4 por 0 o 1.

In [ ]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2
from sklearn.preprocessing import scale, MinMaxScaler
import missingno as msno
import math

In [ ]:
# Leer los datos de cáncer de mama desde UCI (asumiendo que ya están descargados)
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/breast-cancer-wisconsin.data"
column_names = ['Sample code number', 'Clump Thickness', 'Uniformity of Cell Size',
                'Uniformity of Cell Shape', 'Marginal Adhesion',
                'Single Epithelial Cell Size', 'Bare Nuclei', 'Bland Chromatin',
                'Normal Nucleoli', 'Mitoses', 'Class']

data = pd.read_csv(url, names=column_names)

# Reemplazar '?' por NaN en la columna 'Bare Nuclei'
data['Bare Nuclei'] = pd.to_numeric(data['Bare Nuclei'], errors='coerce')

# Imputar valores faltantes en 'Bare Nuclei' con la mediana
median_bare_nuclei = data['Bare Nuclei'].median()
data['Bare Nuclei'].fillna(median_bare_nuclei, inplace=True)

# Cambiar las categorías de 'Class' de 2/4 a 0/1
data['Class'] = data['Class'].replace({2: 0, 4: 1})

# Eliminar la columna 'Sample code number' ya que no es relevante para el análisis
data.drop('Sample code number', axis=1, inplace=True)

data.head(15)

<ipython-input-3-98f81c55077d>:15: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['Bare Nuclei'].fillna(median_bare_nuclei, inplace=True)


,Clump Thickness,Uniformity of Cell Size,Uniformity of Cell Shape,Marginal Adhesion,Single Epithelial Cell Size,Bare Nuclei,Bland Chromatin,Normal Nucleoli,Mitoses,Class
0,5,1,1,1,2,1.0,3,1,1,0
1,5,4,4,5,7,10.0,3,2,1,0
2,3,1,1,1,2,2.0,3,1,1,0
3,6,8,8,1,3,4.0,3,7,1,0
4,4,1,1,3,2,1.0,3,1,1,0
5,8,10,10,8,7,10.0,9,7,1,1
6,1,1,1,1,2,10.0,3,1,1,0
7,2,1,2,1,2,1.0,3,1,1,0
8,2,1,1,1,2,1.0,1,1,5,0
9,4,2,1,1,2,1.0,2,1,1,0


## B.- Realice una detección de valores atípicos univariados por medio del método del rango intercuartílico con 3 de longitud a la derecha y 3 a la izquierda, y elimínelos. Además, realice una detección de valores atípicos multivariados por medio de las distancias de Mahalanobis y elimine aquellos valores que superen el valor de 30.

In [ ]:
# Función para detectar outliers usando el rango intercuartílico
def rango_inter(data, var):
    Q1 = data[var].quantile(0.25)
    Q3 = data[var].quantile(0.75)
    IQR = Q3 - Q1
    lim_inf = Q1 - 3*IQR
    lim_sup = Q3 + 3*IQR
    lista = data.index[(data[var] < lim_inf) | (data[var] > lim_sup)]
    return lista

# Lista de variables predictoras (todas excepto 'Class')
predictors = [col for col in data.columns if col != 'Class']

# Encontrar índices de outliers para todas las variables predictoras
lista_indices = []
for var in predictors:
    lista_indices.extend(rango_inter(data, var))

# Eliminar duplicados y ordenar
lista_indices = sorted(set(lista_indices))

# Eliminar los outliers encontrados
data_limpia = data.drop(lista_indices)

data.head(15)

,Clump Thickness,Uniformity of Cell Size,Uniformity of Cell Shape,Marginal Adhesion,Single Epithelial Cell Size,Bare Nuclei,Bland Chromatin,Normal Nucleoli,Mitoses,Class
0,5,1,1,1,2,1.0,3,1,1,0
1,5,4,4,5,7,10.0,3,2,1,0
2,3,1,1,1,2,2.0,3,1,1,0
3,6,8,8,1,3,4.0,3,7,1,0
4,4,1,1,3,2,1.0,3,1,1,0
5,8,10,10,8,7,10.0,9,7,1,1
6,1,1,1,1,2,10.0,3,1,1,0
7,2,1,2,1,2,1.0,3,1,1,0
8,2,1,1,1,2,1.0,1,1,5,0
9,4,2,1,1,2,1.0,2,1,1,0


In [ ]:
# Calcular distancias de Mahalanobis
def mahalanobis_distances(df):
    # Seleccionar solo las variables numéricas
    X = df[predictors].values

    # Calcular matriz de covarianza
    cov = np.cov(X, rowvar=False)

    # Usar pseudoinversa en lugar de la inversa si la matriz es singular
    try:
        inv_cov = np.linalg.inv(cov)
    except np.linalg.LinAlgError:
        inv_cov = np.linalg.pinv(cov)  # Usar pseudoinversa si es singular

    # Calcular medias
    mean = np.mean(X, axis=0)

    # Calcular distancias de Mahalanobis
    dif = X - mean
    distances = np.sqrt(np.sum(dif @ inv_cov * dif, axis=1))

    return distances

# Calcular distancias para los datos limpios
mahalanobis_dist = mahalanobis_distances(data_limpia)

# Añadir las distancias al dataframe
data_limpia['distMahalanobis'] = mahalanobis_dist

# Identificar outliers (valores mayores a 30)
outliers_maha = data_limpia[data_limpia['distMahalanobis'] > 30].index

# Eliminar los outliers multivariados
data_final = data_limpia.drop(outliers_maha)

# Eliminar la columna de distancias que ya no necesitamos
data_final = data_final.drop('distMahalanobis', axis=1)

data.head(15)

,Clump Thickness,Uniformity of Cell Size,Uniformity of Cell Shape,Marginal Adhesion,Single Epithelial Cell Size,Bare Nuclei,Bland Chromatin,Normal Nucleoli,Mitoses,Class
0,5,1,1,1,2,1.0,3,1,1,0
1,5,4,4,5,7,10.0,3,2,1,0
2,3,1,1,1,2,2.0,3,1,1,0
3,6,8,8,1,3,4.0,3,7,1,0
4,4,1,1,3,2,1.0,3,1,1,0
5,8,10,10,8,7,10.0,9,7,1,1
6,1,1,1,1,2,10.0,3,1,1,0
7,2,1,2,1,2,1.0,3,1,1,0
8,2,1,1,1,2,1.0,1,1,5,0
9,4,2,1,1,2,1.0,2,1,1,0


## C.- Realice un cálculo de los principales estadísticos descriptivos para las variables predictoras y cree dos conjuntos de datos: uno en donde se aplique la transformación Z-score y otro en donde se aplique la normalización mín-máx, sobre todas las variables predictoras.

In [ ]:
# Configurar opciones de visualización
pd.set_option('display.width', 100)
pd.set_option('display.precision', 2)

# Calcular estadísticos descriptivos
stats = data_final[predictors].describe().transpose()
stats['IQR'] = stats['75%'] - stats['25%']
print(stats)

                             count  mean   std  min  25%  50%  75%   max  IQR
Clump Thickness              579.0  3.85  2.53  1.0  2.0  3.0  5.0  10.0  3.0
Uniformity of Cell Size      579.0  2.45  2.59  1.0  1.0  1.0  3.0  10.0  2.0
Uniformity of Cell Shape     579.0  2.58  2.55  1.0  1.0  1.0  3.0  10.0  2.0
Marginal Adhesion            579.0  2.22  2.32  1.0  1.0  1.0  3.0  10.0  2.0
Single Epithelial Cell Size  579.0  2.75  1.81  1.0  2.0  2.0  3.0  10.0  1.0
Bare Nuclei                  579.0  2.72  3.15  1.0  1.0  1.0  3.0  10.0  2.0
Bland Chromatin              579.0  3.00  2.20  1.0  1.5  2.0  3.0  10.0  1.5
Normal Nucleoli              579.0  2.18  2.49  1.0  1.0  1.0  2.0  10.0  1.0
Mitoses                      579.0  1.00  0.00  1.0  1.0  1.0  1.0   1.0  0.0


In [ ]:
# Crear copia del dataframe para la transformación Z-score
data_zscore = data_final.copy()

# Aplicar transformación Z-score a todas las variables predictoras
for var in predictors:
    data_zscore[var] = scale(data_zscore[var])

# Verificar resultados
print("\nDatos con transformación Z-score:")
print(data_zscore[predictors].head())


Datos con transformación Z-score:
   Clump Thickness  Uniformity of Cell Size  Uniformity of Cell Shape  Marginal Adhesion  \
0             0.45                    -0.56                     -0.62              -0.53   
1             0.45                     0.60                      0.56               1.20   
2            -0.34                    -0.56                     -0.62              -0.53   
3             0.85                     2.15                      2.13              -0.53   
4             0.06                    -0.56                     -0.62               0.34   

   Single Epithelial Cell Size  Bare Nuclei  Bland Chromatin  Normal Nucleoli  Mitoses  
0                        -0.42        -0.55        -1.57e-03            -0.48      0.0  
1                         2.35         2.31        -1.57e-03            -0.07      0.0  
2                        -0.42        -0.23        -1.57e-03            -0.48      0.0  
3                         0.14         0.41        -1.57

In [ ]:

# Crear copia del dataframe para la transformación min-max
data_minmax = data_final.copy()

# Aplicar transformación min-max (escalando entre 0 y 1)
scaler = MinMaxScaler(feature_range=(0, 1))
data_minmax[predictors] = scaler.fit_transform(data_minmax[predictors])

# Verificar resultados
print("\nDatos con transformación min-max:")
print(data_minmax[predictors].head())


Datos con transformación min-max:
   Clump Thickness  Uniformity of Cell Size  Uniformity of Cell Shape  Marginal Adhesion  \
0             0.44                     0.00                      0.00               0.00   
1             0.44                     0.33                      0.33               0.44   
2             0.22                     0.00                      0.00               0.00   
3             0.56                     0.78                      0.78               0.00   
4             0.33                     0.00                      0.00               0.22   

   Single Epithelial Cell Size  Bare Nuclei  Bland Chromatin  Normal Nucleoli  Mitoses  
0                         0.11         0.00             0.22             0.00      0.0  
1                         0.67         1.00             0.22             0.11      0.0  
2                         0.11         0.11             0.22             0.00      0.0  
3                         0.22         0.33             